# Proyecto LSP — Pipeline completo en Colab

Notebook end-to-end del proyecto **Reconocimiento de Lengua de Señas Peruana** (UPAO, Percepción Computacional 2026-1).

Lo que hace este notebook en una sola corrida:

1. Extrae los `.rar` del dataset VideoLSP10 (10 clases, ~600 secuencias).
2. Procesa los frames con **MediaPipe Holistic** → vectores `(30, 258)`.
3. Guarda los `.npy` en tu Google Drive.
4. Entrena un **baseline SVM** (Sección 4.1 del .docx).
5. Entrena el modelo **LSTM** (Secciones 4.2 y 4.3).
6. Evalúa con matriz de confusión y classification report (Sección 5).
7. Guarda `modelo_lsp_final.keras` en Drive.

**Pre-requisitos:**
- Activar **GPU T4**: menú `Entorno de ejecución` → `Cambiar tipo de hardware` → `T4 GPU` → Guardar.
- Subir los 6 archivos `rgb.partN.rar` a una carpeta en tu Google Drive.

## Setup — instalar dependencias y montar Drive

In [ ]:
!apt-get install -y unrar -qq
!pip install -q mediapipe

import tensorflow as tf
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

from google.colab import drive
drive.mount('/content/drive')

## Paso 1 — Configuración de rutas y clases

Ajusta `RAR_DIR` con la carpeta de tu Drive donde están los 6 `.rar`.

In [ ]:
from pathlib import Path

# === AJUSTA ESTAS RUTAS SEGUN TU DRIVE ===
RAR_DIR = Path('/content/drive/MyDrive/VideoLSP10_rar')
PROCESSED_DIR = Path('/content/drive/MyDrive/VideoLSP10_processed')
MODELS_DIR = Path('/content/drive/MyDrive/VideoLSP10_models')

# Carpetas temporales en disco local de Colab (rapidas, se borran al cerrar)
LOCAL_RAR = Path('/content/rar_files')
LOCAL_EXTRACTED = Path('/content/rgb_extracted')

# Configuracion del modelo
CLASSES = [
    'ayudame', 'por_favor', 'disculpame', 'cual_es_tu_nombre',
    'donde_vives_tu', 'no_entiendo', 'que_haces_tu',
    'hola_como_estas_tu', 'gracias', 'hasta_manana',
]
NUM_CLASSES = len(CLASSES)
FRAMES = 30
FEATURES_PER_FRAME = 258  # pose(132) + mano_izq(63) + mano_der(63)

# Crear directorios
for d in (PROCESSED_DIR, MODELS_DIR, LOCAL_RAR, LOCAL_EXTRACTED):
    d.mkdir(parents=True, exist_ok=True)

# Verificar que los .rar esten en Drive
rars = sorted(RAR_DIR.glob('rgb.part*.rar'))
print(f'Encontrados {len(rars)} archivos .rar en {RAR_DIR}:')
for r in rars:
    print(f'  - {r.name} ({r.stat().st_size / 1024 / 1024:.0f} MB)')

assert len(rars) == 6, 'Deberias tener exactamente 6 archivos rgb.partN.rar'

## Paso 2 — Extraer los `.rar` en disco local de Colab

Copia los `.rar` de Drive al disco temporal (más rápido para leer), luego los descomprime. Tarda 5-15 minutos.

In [ ]:
import shutil

print('Copiando .rar de Drive a disco local de Colab...')
for r in rars:
    destino = LOCAL_RAR / r.name
    if not destino.exists():
        shutil.copy(r, destino)
        print(f'  copiado: {r.name}')
    else:
        print(f'  ya estaba: {r.name}')

print('\nExtrayendo (5-15 min)...')
!unrar x -o+ "{LOCAL_RAR}/rgb.part1.rar" "{LOCAL_EXTRACTED}/" > /tmp/unrar.log 2>&1 && echo '   OK' || (echo '   ERROR:'; tail -20 /tmp/unrar.log)

In [ ]:
# Verificar estructura extraida
from collections import Counter

def parse_folder(name):
    parts = name.rsplit('_r.', 1)
    if len(parts) != 2:
        return None, None
    try:
        return parts[0], int(parts[1])
    except ValueError:
        return None, None

# Algunas extracciones dejan todo bajo una subcarpeta, buscar el nivel correcto
candidatos = [LOCAL_EXTRACTED] + [p for p in LOCAL_EXTRACTED.iterdir() if p.is_dir()]
BASE_RGB = None
for c in candidatos:
    sub = [p for p in c.iterdir() if p.is_dir() and parse_folder(p.name)[0] in CLASSES]
    if sub:
        BASE_RGB = c
        break

assert BASE_RGB is not None, 'No se encontraron carpetas <clase>_r.<id>'
print(f'Carpeta base con secuencias: {BASE_RGB}')

carpetas = sorted([p for p in BASE_RGB.iterdir() if p.is_dir()])
conteo = Counter()
for c in carpetas:
    clase, _ = parse_folder(c.name)
    if clase:
        conteo[clase] += 1

print(f'\nTotal secuencias: {sum(conteo.values())}')
print('Muestras por clase:')
for clase in CLASSES:
    print(f'  {clase}: {conteo.get(clase, 0)}')

ejemplo = next(c for c in carpetas if parse_folder(c.name)[0] in CLASSES)
n_frames = len(list(ejemplo.glob('*.jpg')))
print(f'\nEjemplo: {ejemplo.name} tiene {n_frames} frames')

## Paso 3 — Extraer keypoints con MediaPipe Holistic

Por cada secuencia:
1. Lee los frames `.jpg`.
2. Muestrea uniformemente a 30 frames.
3. Cada frame → pose(132) + mano_izq(63) + mano_der(63) = vector de 258.
4. Guarda como `(30, 258)` en `Drive/VideoLSP10_processed/<clase>/<id>.npy`.

**Es resumible**: si Colab se desconecta, al re-ejecutar continúa donde quedó (salta los `.npy` ya hechos). Tarda 10-20 minutos.

In [ ]:
import numpy as np
import cv2
import mediapipe as mp
from tqdm import tqdm

mp_holistic = mp.solutions.holistic


def extraer_keypoints(resultados):
    pose = (
        np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in resultados.pose_landmarks.landmark]).flatten()
        if resultados.pose_landmarks else np.zeros(33 * 4)
    )
    lh = (
        np.array([[lm.x, lm.y, lm.z] for lm in resultados.left_hand_landmarks.landmark]).flatten()
        if resultados.left_hand_landmarks else np.zeros(21 * 3)
    )
    rh = (
        np.array([[lm.x, lm.y, lm.z] for lm in resultados.right_hand_landmarks.landmark]).flatten()
        if resultados.right_hand_landmarks else np.zeros(21 * 3)
    )
    return np.concatenate([pose, lh, rh])


def procesar_secuencia(carpeta, holistic):
    archivos = sorted(carpeta.glob('*.jpg'), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
    n = len(archivos)
    if n == 0:
        return None
    indices = np.linspace(0, n - 1, FRAMES).astype(int)
    secuencia = []
    for i in indices:
        img = cv2.imread(str(archivos[i]))
        if img is None:
            secuencia.append(np.zeros(FEATURES_PER_FRAME))
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        resultados = holistic.process(rgb)
        secuencia.append(extraer_keypoints(resultados))
    return np.array(secuencia, dtype=np.float32)


# Preparar tareas (skip las ya procesadas)
for clase in CLASSES:
    (PROCESSED_DIR / clase).mkdir(parents=True, exist_ok=True)

tareas = []
for c in carpetas:
    clase, idx = parse_folder(c.name)
    if clase not in CLASSES:
        continue
    salida = PROCESSED_DIR / clase / f'{idx}.npy'
    if not salida.exists():
        tareas.append((c, clase, idx, salida))

print(f'Pendientes: {len(tareas)} secuencias')

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for carpeta, clase, idx, salida in tqdm(tareas, desc='MediaPipe'):
        secuencia = procesar_secuencia(carpeta, holistic)
        if secuencia is not None and secuencia.shape == (FRAMES, FEATURES_PER_FRAME):
            np.save(salida, secuencia)

print('\nExtraccion de keypoints completa.')

## Paso 4 — Cargar dataset `.npy` y split estratificado 80/10/10

In [ ]:
from sklearn.model_selection import train_test_split

X, y = [], []
for idx, clase in enumerate(CLASSES):
    for archivo in sorted((PROCESSED_DIR / clase).glob('*.npy')):
        X.append(np.load(archivo))
        y.append(idx)

X = np.array(X, dtype=np.float32)
y = np.array(y)
print(f'Dataset cargado: X={X.shape}, y={y.shape}')

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=1/9, stratify=y_tmp, random_state=42)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Paso 5 — Baseline SVM (Sección 4.1)

Features estadísticas (media + std + rango) por landmark → 774 features → SVM con kernel RBF.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


def extraer_caracteristicas_estadisticas(secuencia):
    # secuencia: array de forma (30, 258)
    media = np.mean(secuencia, axis=0)
    std   = np.std(secuencia, axis=0)
    rango = np.max(secuencia, axis=0) - np.min(secuencia, axis=0)
    return np.concatenate([media, std, rango])  # vector de 774 features


X_train_stats = np.array([extraer_caracteristicas_estadisticas(s) for s in X_train])
X_test_stats  = np.array([extraer_caracteristicas_estadisticas(s) for s in X_test])

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_stats)
X_test_sc  = scaler.transform(X_test_stats)

svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm.fit(X_train_sc, y_train)
precision_baseline = accuracy_score(y_test, svm.predict(X_test_sc))
print(f'Precisión Baseline SVM: {precision_baseline:.2%}')

## Paso 6 — Entrenar modelo LSTM (Secciones 4.2 y 4.3)

Arquitectura: `LSTM 128 → Dropout → LSTM 64 → Dropout → Dense 64 → Dense 10 (Softmax)`.

Tarda ~10-20 minutos en GPU T4.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

model = Sequential([
    Input(shape=(FRAMES, FEATURES_PER_FRAME)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax'),
])

# Compilacion del modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ModelCheckpoint('modelo_lsp_final.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
]

# Entrenamiento
historial = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## Paso 7 — Evaluación (Sección 5)

Curvas de aprendizaje + classification report + matriz de confusión.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Curvas de loss y accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historial.history['loss'], label='train')
axes[0].plot(historial.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(historial.history['accuracy'], label='train')
axes[1].plot(historial.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('curvas_aprendizaje.png', dpi=120, bbox_inches='tight')
plt.show()

# Predicciones en test
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=CLASSES))

precision_lstm = (y_pred == y_test).mean()
print(f'\n=== Comparacion ===')
print(f'  Baseline SVM: {precision_baseline:.2%}')
print(f'  Modelo LSTM:  {precision_lstm:.2%}')

# Matriz de confusion
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusion - LSTM')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=color, fontsize=10)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

## Paso 8 — Guardar todo en Drive y descargar el modelo

In [ ]:
import shutil

modelo_final = MODELS_DIR / 'modelo_lsp_final.keras'
shutil.copy('modelo_lsp_final.keras', modelo_final)
shutil.copy('curvas_aprendizaje.png', MODELS_DIR / 'curvas_aprendizaje.png')
shutil.copy('matriz_confusion.png',  MODELS_DIR / 'matriz_confusion.png')

print(f'Modelo y graficas guardados en: {MODELS_DIR}')
print('\nDescargando modelo a tu PC...')

from google.colab import files
files.download(str(modelo_final))